<a id="inizio"></a>
# 08 · Pandas: le date · Soluzioni

⏱ ~80 min &nbsp;·&nbsp; Dati: `../Dati/load_total_north_hourly_2024.xlsx`

**In questo notebook impariamo a**

- convertire testo in date con `pd.to_datetime` ed estrarne ora, giorno e mese
- usare il tempo come indice per trovare i buchi e cambiare granularità con `resample`
- costruire lag e medie mobili con `shift` e `rolling` e gestire l'ora legale

<a id="indice"></a>
**Indice**

- [1. Dataset toy](#sez-1)
- [2. `datetime` in pandas](#sez-2)
- [3. Formati](#sez-3)
- [4. Il tempo come indice](#sez-4)
- [5. Attributi temporali](#sez-5)
- [6. Frequenza](#sez-6)
- [7. Missing che emergono con l'allineamento](#sez-7)
- [8. Resampling](#sez-8)
- [9. Shift e rolling](#sez-9)
- [10. Differenze tra date](#sez-10)
- [11. Fuso orario e ora legale](#sez-11)
    - [11.1 I duplicati dell'ora legale nei dati Terna](#sez-11-1)
- [12. Esercizi](#sez-12)
    - [Esercizio 8.1 · Il carico giornaliero](#es-8-1)
    - [Esercizio 8.2 · Ora e giorno della settimana](#es-8-2)
    - [Esercizio 8.3 · La media mobile su 24 ore (facoltativo)](#es-8-3)

<a id="sez-1"></a>
## 1. Dataset toy

Costruiamo un dataset piccolo ma realistico: misure quartorarie di potenza (`power_kw`) e temperatura (`temp_c`).
Il timestamp è intenzionalmente una **stringa** in formato italiano (`DD/MM/YYYY HH:MM`).
Poi inseriamo qualche buco per simulare acquisizioni mancanti.

In [ ]:
import numpy as np
np.random.seed(42)
import pandas as pd

pd.set_option("display.max_rows", 10)
pd.set_option("display.max_columns", 20)

In [ ]:
rng = pd.date_range("2025-03-01 00:00", periods=3 * 24 * 4, freq="15min")

# pattern giornaliero semplice + rumore: giusto per avere una serie "viva"
hours = rng.hour + rng.minute / 60
power_kw = 220 + 60 * np.sin(2 * np.pi * (hours / 24)) + np.random.normal(0, 8, size=len(rng))
temp_c = 12 + 5 * np.sin(2 * np.pi * ((hours - 6) / 24)) + np.random.normal(0, 0.7, size=len(rng))

df = pd.DataFrame({
    "timestamp_str": rng.strftime("%d/%m/%Y %H:%M"),  # formato tipico in Italia
    "power_kw": power_kw.round(1),
    "temp_c": temp_c.round(1),
})

# rimuoviamo alcune righe per simulare missing (buchi nella serie)
drop_idx = np.random.choice(df.index, size=18, replace=False)  # ~2.5% su 288 punti
df = df.drop(drop_idx).reset_index(drop=True)

df.head()

`df.dtypes` conferma che `timestamp_str` è testo (`str`).

In [ ]:
df.dtypes

<a id="sez-2"></a>
## 2. `datetime` in pandas

Una colonna di testo che "sembra una data" resta testo. `pd.to_datetime` la trasforma in un tipo
temporale (`datetime64`) che si può ordinare, filtrare per periodo e usare per calcolare intervalli.

Documentazione: [`pd.to_datetime`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.to_datetime.html)

In [ ]:
df["timestamp"] = pd.to_datetime(df["timestamp_str"], dayfirst=True)
df[["timestamp_str", "timestamp"]].head()

In [ ]:
df.dtypes

> **Nota.**
>
> In Italia il caso più comune è `giorno/mese/anno`, quindi `dayfirst=True` è spesso la scelta corretta.
> Quando il formato è noto e stabile, `format=...` rende la conversione più affidabile.

<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Prova tu**

Crea una nuova colonna `timestamp_2` convertendo di nuovo `timestamp_str`, ma questa volta con
`format="%d/%m/%Y %H:%M"`. Confronta le prime 5 righe con `timestamp`: devono coincidere.

</div>

In [ ]:
df["timestamp_2"] = pd.to_datetime(df["timestamp_str"], format="%d/%m/%Y %H:%M")
df[["timestamp", "timestamp_2"]].head()

<a id="sez-3"></a>
## 3. Formati

Il formato serve soprattutto quando la stringa è **ambigua** o non standard.
Esempio classico: `01/02/2025` può essere 1 febbraio oppure 2 gennaio (dipende dal contesto).

In [ ]:
# esempio di ambiguità giorno/mese + gestione errori
ambigue = pd.Series(["01/02/2025 08:00", "13/02/2025 08:00"])

parsed_dayfirst = pd.to_datetime(ambigue, dayfirst=True)
parsed_monthfirst = pd.to_datetime(ambigue, dayfirst=False, errors="coerce")  # qui un valore diventa NaT

pd.DataFrame({
    "stringa": ambigue,
    "dayfirst=True": parsed_dayfirst,
    "dayfirst=False (coerce)": parsed_monthfirst,
})

Senza `errors="coerce"` vale il default, `errors="raise"`: la conversione si ferma con un errore
sul primo valore che non torna. Questa cella dà errore apposta.

In [ ]:
pd.to_datetime(ambigue, dayfirst=False, errors="raise")

<a id="sez-4"></a>
## 4. Il tempo come indice

Quando il timestamp è l'indice, pandas abilita selezioni "per periodo" senza calcoli manuali.
Prima di tutto: ordinare l'indice. Con dati reali capita spesso di avere righe fuori ordine.

Documentazione: [serie temporali, user guide](https://pandas.pydata.org/pandas-docs/stable/user_guide/timeseries.html)

In [ ]:
df_ts = (
    df[["timestamp", "power_kw", "temp_c"]]
    .set_index("timestamp")
    .sort_index()
)

df_ts.head()

In [ ]:
# slicing per periodo (anno/mese/giorno)
df_ts.loc["2025-03-02"].head()

<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Prova tu**

Seleziona in `periodo` le righe dal `2025-03-01 12:00` al `2025-03-01 18:00` usando `.loc[...]`.
Sono 22 righe: in quelle sei ore ci sono tre buchi.

</div>

In [ ]:
periodo = df_ts.loc["2025-03-01 12:00":"2025-03-01 18:00"]
len(periodo)  # Output: 22

<a id="sez-5"></a>
## 5. Attributi temporali

Una volta che il tempo è `datetime`, estrarre componenti temporali diventa semplice (mese, ora, giorno della settimana).
Se il tempo è indice, si lavora via `df.index`; se è una colonna, si usa `.dt`.

Documentazione: [accessor `.dt`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.Series.dt.html)

In [ ]:
df_feat = df_ts.copy()
df_feat["hour"] = df_feat.index.hour
df_feat["dayofweek"] = df_feat.index.dayofweek  # 0=lunedì
df_feat["month"] = df_feat.index.month

df_feat.head()

<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Prova tu**

Aggiungi a `df_feat_ex` le colonne `date` (solo data) e `is_weekend` (`True` per sabato e domenica).
Il 1° marzo 2025 è un sabato: le righe del weekend sono 182.

Suggerimento: `df_feat_ex.index.date` e `dayofweek` (sabato=5, domenica=6).

</div>

In [ ]:
df_feat_ex = df_feat.copy()
df_feat_ex["date"] = df_feat_ex.index.date
df_feat_ex["is_weekend"] = df_feat_ex["dayofweek"] >= 5

df_feat_ex["is_weekend"].sum()  # Output: 182

<a id="sez-6"></a>
## 6. Frequenza

"Frequenza" significa intervallo atteso tra due timestamp consecutivi (15 minuti, 1 ora, 1 giorno).
Con dati puliti e regolari, pandas può inferirla; con buchi o irregolarità spesso no.
Frequenze comuni: `15min` (quartorario), `h` (orario), `D` (giornaliero), `MS` (inizio mese).

Documentazione: [alias delle frequenze](https://pandas.pydata.org/pandas-docs/stable/user_guide/timeseries.html#dateoffset-objects)

In [ ]:
df_ts.index.inferred_freq

Non restituisce niente: con i buchi pandas non riesce a inferire una frequenza.
La differenza tra timestamp consecutivi mostra cosa succede.

In [ ]:
df_ts.index.to_series().diff().value_counts().head()

<a id="sez-7"></a>
## 7. Missing che emergono con l'allineamento

Quando "forzi" una frequenza regolare, i timestamp mancanti diventano righe con `NaN`.
Questo è utile: rende visibili buchi che altrimenti restano nascosti.
Qui usiamo `asfreq` per allineare a frequenza quartoraria (`15min`).

Documentazione: [`DataFrame.asfreq`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.asfreq.html)

In [ ]:
df_qh = df_ts.asfreq("15min")
df_qh

In [ ]:
missing_counts = df_qh.isna().sum()
missing_counts

In [ ]:
# dove mancano i valori di power_kw?
missing_timestamps = df_qh.index[df_qh["power_kw"].isna()]
missing_timestamps[:10]

I grafici qui sotto anticipano il notebook su Plotly: per ora si leggono e basta.

In [ ]:
# i dati prima e dopo asfreq: nel secondo grafico i buchi interrompono la linea
import plotly.express as px

fig = px.line(df_ts, x=df_ts.index, y="power_kw", title="Dati originali (con buchi)")
fig.show()
fig = px.line(df_qh, x=df_qh.index, y="power_kw", title="Dati con asfreq (buchi evidenziati)")
fig.show()

Ora che abbiamo individuato i valori mancanti, dobbiamo decidere come gestirli. Due approcci semplici e comuni.

**Interpolazione lineare**: pandas collega il valore prima e dopo il buco con una retta. Se alle 10:00
abbiamo 200 kW, alle 11:00 220 kW e manca il valore delle 10:30, stima 210 kW. Utile quando il fenomeno
cambia gradualmente (una temperatura). Comando: `df.interpolate(method="linear")`.

**Forward fill** (riempimento in avanti): copia l'ultimo valore valido nei buchi successivi. Utile
quando il valore resta stabile per un po' (lo stato di un dispositivo). Comando: `df.ffill()`.

**Backward fill**: `df.bfill()` fa il contrario, copia all'indietro il prossimo valore valido.

In [ ]:
# lavoriamo SOLO su power_kw
s = df_qh["power_kw"]

t0 = missing_timestamps[0]
# finestra: da t0-45min a t0+45min
idx = slice(t0 - pd.Timedelta(minutes=45), t0 + pd.Timedelta(minutes=45))

# imputazioni (una colonna ciascuna)
out = pd.DataFrame({
    "original": s.loc[idx],
    "time": s.interpolate(method="time").loc[idx],
    "linear": s.interpolate(method="linear").loc[idx],
    "ffill": s.ffill().loc[idx],
    "bfill": s.bfill().loc[idx],
})

out

I metodi `time` e `linear` stimano il valore mancante interpolando tra i due punti vicini.
Con frequenza regolare, i due metodi danno lo stesso risultato (qui 271.25).

In [ ]:
# visualizzazione con Plotly delle imputazioni
import plotly.graph_objects as go

fig = go.Figure()
fig.add_trace(go.Scatter(x=out.index, y=out["time"], mode="markers+lines", name="interpolate time"))
fig.add_trace(go.Scatter(x=out.index, y=out["linear"], mode="markers+lines", name="interpolate linear"))
fig.add_trace(go.Scatter(x=out.index, y=out["ffill"], mode="markers+lines", name="ffill"))
fig.add_trace(go.Scatter(x=out.index, y=out["bfill"], mode="markers+lines", name="bfill"))
fig.update_layout(title="Imputazioni per missing value", height=400, width=700)
fig.show()

<a id="sez-8"></a>
## 8. Resampling

Il resampling cambia granularità temporale: da quartorario a orario, giornaliero, mensile.
È un'operazione di aggregazione guidata dal tempo (non dalla posizione delle righe).

Documentazione: [`resample`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.resample.html)

Prima riempiamo i buchi con l'interpolazione lineare.

In [ ]:
df_qh = df_qh.interpolate(method="linear")

In [ ]:
hourly = df_qh.resample("h").mean()
daily = df_qh.resample("D").mean()

hourly.head()

In [ ]:
daily

<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Prova tu**

Crea un resampling **orario** di `power_kw` con la somma (`sum`) invece della media, e confronta
le prime righe con `hourly`. La somma è 4 volte la media: in ogni ora ci sono 4 quarti d'ora.

</div>

In [ ]:
hourly_sum = df_qh["power_kw"].resample("h").sum()
pd.DataFrame({"mean": hourly["power_kw"], "sum": hourly_sum}).head()

<a id="sez-9"></a>
## 9. Shift e rolling

Due operazioni semplici e molto usate nella gestione di time series:

- `shift(k)`: sposta i valori di `k` step nel tempo (crea "lag").
- `rolling(w)`: calcola statistiche su una finestra mobile di ampiezza `w`.

Entrambe possono introdurre `NaN` in testa (perché mancano i valori "precedenti").
Documentazione: [`rolling`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.rolling.html)

In [ ]:
df_sr = df_qh.copy()

# lag di 1 ora: con frequenza 15min corrisponde a 4 step
df_sr["power_lag_4"] = df_sr["power_kw"].shift(4)

# media mobile su 2 ore: 8 step
df_sr["power_ma_8"] = df_sr["power_kw"].rolling(8).mean()

df_sr[["power_kw", "power_lag_4", "power_ma_8"]].head(12)

In [ ]:
# shift di 1 ora (4 step) contro power_kw
fig = px.line(df_sr, x=df_sr.index, y=["power_kw", "power_lag_4"], title="Shift di 1 ora (4 step)")
fig.show()

In [ ]:
# media mobile su una finestra di 2 ore (8 step) contro power_kw
fig = px.line(df_sr, x=df_sr.index, y=["power_kw", "power_ma_8"], title="Media mobile su finestra di 2 ore (8 step)")
fig.show()

<a id="sez-10"></a>
## 10. Differenze tra date

Le differenze tra timestamp producono `Timedelta` (durate).
Servono sia per capire la regolarità della serie sia per misurare intervalli (es. "quante ore copre il dataset?").

In [ ]:
# differenze tra timestamp consecutivi
delta_consecutivi = df_qh.index.to_series().diff()

delta_consecutivi.head(10)

In [ ]:
# intervallo totale coperto
delta_totale = df_qh.index.max() - df_qh.index.min()
delta_totale

In [ ]:
# durata in ore (float)
delta_totale.total_seconds() / 3600

<a id="sez-11"></a>
## 11. Fuso orario e ora legale

Un timestamp può essere **naive**, senza fuso orario (solo data e ora), oppure **timezone-aware**,
con fuso orario (es. `Europe/Rome`). Spesso si ricevono orari locali naive da interpretare correttamente.
L'ora legale è la trappola tipica: alcune ore non esistono (primavera) o si ripetono (autunno).

Documentazione: [fusi orari](https://pandas.pydata.org/pandas-docs/stable/user_guide/timeseries.html#time-zone-handling)

<div style="background:rgba(217,83,79,0.12); border-left:4px solid #d9534f; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**⚠️ Attenzione**

Localizzare un indice che attraversa un cambio ora legale può generare timestamp ambigui o inesistenti:
in quei casi servono i parametri `ambiguous=` e `nonexistent=`.
Se i dati arrivano già in UTC, spesso è meglio mantenerli in UTC e convertire solo in visualizzazione.

</div>

In [ ]:
df_tz = df_qh.copy()

# interpretiamo i timestamp come orari locali italiani
df_tz.index = df_tz.index.tz_localize("Europe/Rome")

# conversione a UTC (utile per sistemi e confronti)
df_utc = df_tz.tz_convert("UTC")

df_tz.index[:3], df_utc.index[:3]

In [ ]:
df_tz.loc["2025-03-01 06:00":"2025-03-01 12:00"].head()

<a id="sez-11-1"></a>
### 11.1 I duplicati dell'ora legale nei dati Terna

Nei dati veri il cambio d'ora si vede. Il carico della zona Nord 2024 di Terna è in ora locale, un valore
ogni 15 minuti (il nome del file dice `hourly`), con le righe dall'ultima alla prima. Le rimettiamo in
ordine con `sort_values` e contiamo i timestamp duplicati.

In [ ]:
carico = pd.read_excel("../Dati/load_total_north_hourly_2024.xlsx")
carico = carico.sort_values("Date")
carico["Date"].duplicated().sum()

In [ ]:
doppioni = carico["Date"].duplicated(keep=False)
carico[doppioni]

Le 2:00, 2:15, 2:30 e 2:45 del 27 ottobre, l'ultima domenica del mese, compaiono due volte con carichi
diversi: una è l'ultima ora legale, l'altra la prima ora solare, e il file non dice quale sia quale.
`drop_duplicates(subset="Date")` tiene la prima occorrenza di ogni timestamp.

In [ ]:
carico = carico.drop_duplicates(subset="Date")
len(carico)

Restano 35.132 righe invece di 35.136 (8784 ore per 4): i quattro quarti d'ora mancanti sono quelli
dell'ora che a marzo non esiste.

<a id="sez-12"></a>
## 12. Esercizi

Gli esercizi usano `carico`, il file Terna già ordinato e senza duplicati. La colonna
`Total Load [MW]` è una potenza media su ogni quarto d'ora: un quarto d'ora a 100 MW sono 25 MWh,
quindi l'energia si ottiene sommando i valori e dividendo per 4.

<a id="es-8-1"></a>
<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Esercizio 8.1 · Il carico giornaliero**

Dal carico quartorario vogliamo l'energia consumata ogni giorno del 2024 nella zona Nord.

**Cosa fare.**

1. Metti in `serie` la colonna `Total Load [MW]` di `carico`, con `Date` come indice (`set_index`).
2. Calcola in `giornaliero` l'energia di ogni giorno in MWh: `resample("D")`, `sum()`, poi diviso 4.
3. Metti in `giorno_max` il giorno con l'energia più alta (`idxmax()`).

*Suggerimento:* si seleziona la colonna prima di `resample`, così la colonna di testo `Bidding Zone` resta fuori.

</div>

<div style="background:rgba(46,159,214,0.12); border-left:4px solid #2e9fd6; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✅ Soluzione 8.1**

</div>

In [ ]:
serie = carico.set_index("Date")["Total Load [MW]"]
giornaliero = serie.resample("D").sum() / 4
giorno_max = giornaliero.idxmax()

print(len(giornaliero))  # Output: 366
giorno_max

In [ ]:
from corso import verifica
verifica("8.1")

<a id="es-8-2"></a>
<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Esercizio 8.2 · Ora e giorno della settimana**

Vogliamo sapere a che ora del giorno e in quale giorno della settimana il carico medio è più alto e più basso.

**Cosa fare.**

1. Aggiungi a `carico` le colonne `ora` (`.dt.hour`) e `giorno_settimana` (`.dt.dayofweek`, 0=lunedì) a partire da `Date`.
2. Calcola in `per_ora` il carico medio per ora con `groupby`, e in `ora_di_punta` l'ora con la media più alta.
3. Calcola in `per_giorno` il carico medio per giorno della settimana, e in `giorno_minimo` il giorno con la media più bassa.

</div>

<div style="background:rgba(46,159,214,0.12); border-left:4px solid #2e9fd6; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✅ Soluzione 8.2**

</div>

In [ ]:
carico["ora"] = carico["Date"].dt.hour
carico["giorno_settimana"] = carico["Date"].dt.dayofweek

per_ora = carico.groupby("ora")["Total Load [MW]"].mean()
ora_di_punta = per_ora.idxmax()

per_giorno = carico.groupby("giorno_settimana")["Total Load [MW]"].mean()
giorno_minimo = per_giorno.idxmin()

print(ora_di_punta, giorno_minimo)

In [ ]:
from corso import verifica
verifica("8.2")

<a id="es-8-3"></a>
<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Esercizio 8.3 · La media mobile su 24 ore (facoltativo)**

Il carico quartorario oscilla molto tra giorno e notte: una media mobile su 24 ore mostra l'andamento di fondo.

**Cosa fare.**

1. Calcola in `media_24h` la media mobile su 24 ore di `Total Load [MW]`, con `Date` come indice.
2. Metti in `picco_24h` il timestamp in cui la media mobile è più alta.

*Suggerimento:* 24 ore sono 96 quarti d'ora.

</div>

<div style="background:rgba(46,159,214,0.12); border-left:4px solid #2e9fd6; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✅ Soluzione 8.3**

</div>

In [ ]:
serie = carico.set_index("Date")["Total Load [MW]"]
media_24h = serie.rolling(96).mean()
picco_24h = media_24h.idxmax()

picco_24h

In [ ]:
from corso import verifica
verifica("8.3")

---

**Fine del notebook 08.** Precedente: [07 · Pandas: operazioni sui DataFrame](07_Pandas_operazioni.ipynb) &nbsp;·&nbsp; Prossimo: [09 · Plotly per serie storiche](09_Plotly.ipynb)

<sub>Blocco 3 · Analisi dei dati con pandas e visualizzazione · Giornata 2 · Aula Avanzata</sub>